In [1]:
## Alle für das Programm nötigen Imports
### Abschnitt 1
import numpy as np
import matplotlib.pyplot as plt
import random
### Abschnitt 2
import ipywidgets as widgets
from IPython.display import display
### Abschnitt 4
import time
import pandas as pd
### Abschnitt 5
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_ibm_runtime import QiskitRuntimeService, Estimator, Sampler
import time
### Abschnitt 6
import pandas as pd
import matplotlib.pyplot as plt




ImportError: cannot import name 'BaseSampler' from 'qiskit.primitives' (c:\Users\juanc\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\Python_JNotebook\Qiskit\qiskit_env\lib\site-packages\qiskit\primitives\__init__.py)

In [ ]:
# Abschnitt 1: Kundenpunkte generieren und Depot definieren

## Einstellungen

NUM_CUSTOMERS = 19 # zusätzlich zum Depot = 20 Punkte insgesamt
MAP_SIZE = 100 # 100x100 km Karte
SEED = 66 # Damit alles wiederholbar bleibt

## Schwierigkeitsmodus: 'einfach', 'schwer', 'cluster'

MODE = 'cluster' # Ändere auf 'einfach', 'schwer' nach Belieben
random.seed(SEED)
np.random.seed(SEED)
def generate_points(mode='einfach'):
if mode == 'einfach':
### Ganz normale Zufallsverteilung im Quadrat
depot = np.array([MAP_SIZE // 2, MAP_SIZE // 2]) # Depot zentral platzieren
points = np.random.uniform(0, MAP_SIZE, (NUM_CUSTOMERS, 2))
elif mode == 'schwer':
### Zufallsverteilung, aber Depot am Rand – die längsten Wege sind nötig
depot = np.array()
points = np.random.uniform(0, MAP_SIZE, (NUM_CUSTOMERS, 2))
elif mode == 'cluster':
### 2 Cluster: Cluster 1 um (15,85), Cluster 2 um (85,15)
depot = np.array([MAP_SIZE // 2, MAP_SIZE // 2])
cluster1 = np.random.normal(loc=, scale=8, size=(NUM_CUSTOMERS//2, 2))
cluster2 = np.random.normal(loc=, scale=8, size=(NUM_CUSTOMERS-NUM_CUSTOMERS//2, 2))
points = np.vstack([cluster1, cluster2])
else:
raise ValueError("Unbekannter Modus.")
### Depot als ersten Eintrag
all_points = np.vstack([depot, points])
return all_points, depot
points, depot = generate_points(MODE)


## Visualisierung: Punkte plotten 

plt.figure(figsize=(8,8))
plt.scatter(points[1:,0], points[1:,1], c='blue', label='Kunden')
plt.scatter(depot, depot, c='red', s=120, label='Depot (Start/Ende)', marker='s')
for i in range(len(points)):
plt.text(points[i,0], points[i,1], str(i), fontsize=9, ha='right', va='bottom')
plt.title(f'Kundenkarte: Modus "{MODE}"')
plt.xlabel('X [km]')
plt.ylabel('Y [km]')
plt.legend()
plt.grid(True)
plt.show()


In [6]:
# Abschnitt 2: Eingabe – Priorisierten Kunden & Zeitlimit auswählen



## Dropdown: Wähle Prioritätskunde (außer 0, weil 0 = Depot) 

prioritized_dropdown = widgets.Dropdown(
options=[(f"Kunde {i}", i) for i in range(1, NUM_CUSTOMERS+1)],
value=1,
description='Priorisierter Kunde:'
)

## Zahleneingabe: Nach wie vielen Minuten soll der priorisierte Kunde spätestens angefahren werden?

zeitfenster_field = widgets.IntText(
value=90,
description='Max Minuten bis Stopp:',
)
display(prioritized_dropdown, zeitfenster_field)

NameError: name 'NUM_CUSTOMERS' is not defined

In [ ]:
# Abschnitt 3: Matrix für Entfernungen & Fahrtzeit bauen

LKW_SPEED = 80 # km/h
MAX_MINUTES = 540 # max Fahrtzeit in Minuten

## Hilfsfunktion: Distanz in km 

def calc_dist(p1, p2):
return np.linalg.norm(np.array(p1)-np.array(p2))
dist_matrix = np.zeros((len(points), len(points)))
for i in range(len(points)):
for j in range(len(points)):
dist_matrix[i, j] = calc_dist(points[i], points[j])

## Zeitmatrix (Minuten) 

time_matrix = dist_matrix / LKW_SPEED * 60

## Für spätere Nutzung in Algorithmen 

print("Distanzmatrix und Zeitmatrix bereit!")


In [ ]:
# Abschnitt 4: Klassischer Algorithmus (z.B. Nearest Neighbor mit Constraint)


def classical_route(points, depot_idx, prioritized_idx, zeitfenster_minutes):
n = len(points)
visited = [False] * n
route = [depot_idx]
visited[depot_idx] = True
total_minutes = 0
customers_before_priority = []
## Hinweg: Finde maximal viele Stops bis zum Prioritätskunden, Zeitrestriktion beachten
current_idx = depot_idx
while True:
    ## Wenn Prioritätskunde als nächstes besucht werden muss (Zeit ist knapp)
    time_to_priority = time_matrix[current_idx, prioritized_idx]
    if total_minutes + time_to_priority > zeitfenster_minutes:
        break
    ## Finde kürzesten Kunden-Stopp, der noch besucht werden kann ohne Zeitverletzung
    candidates = [i for i in range(n) if not visited[i] and i != prioritized_idx]
    if not candidates: break
    next_idx = min(candidates, key=lambda i: time_matrix[current_idx, i])
    route.append(next_idx)
    visited[next_idx] = True
    minutes_to_next = time_matrix[current_idx, next_idx]
    total_minutes += minutes_to_next
    current_idx = next_idx
    customers_before_priority.append(next_idx)
## Jetzt Prioritätskunde anfahren
route.append(prioritized_idx)
visited[prioritized_idx] = True
total_minutes += time_matrix[current_idx, prioritized_idx]
current_idx = prioritized_idx

## Rückweg zum Depot, maximale Kunden mit Restzeit besuchen
customers_after_priority = []
while total_minutes + time_matrix[current_idx, depot_idx] < MAX_MINUTES:
    candidates = [i for i in range(n) if not visited[i] and i != depot_idx]
    if not candidates: break
    next_idx = min(candidates, key=lambda i: time_matrix[current_idx, i])
    if total_minutes + time_matrix[current_idx, next_idx] + time_matrix[next_idx, depot_idx] > MAX_MINUTES:
        break
    route.append(next_idx)
    visited[next_idx] = True
    minutes_to_next = time_matrix[current_idx, next_idx]
    total_minutes += minutes_to_next
    current_idx = next_idx
    customers_after_priority.append(next_idx)
## Zum Depot zurück
route.append(depot_idx)
total_minutes += time_matrix[current_idx, depot_idx]

## Speichern
route_names = [f'Punkt {i}' for i in route]
route_df = pd.DataFrame({'Reihenfolge': route_names, 'Index': route})
route_df.to_csv('classical_route.csv', index=False)
return route, total_minutes, route_df


## Ausführen und messen 

start = time.time()
depot_idx = 0
prioritized_idx = prioritized_dropdown.value
route, total_time, route_df = classical_route(points, depot_idx, prioritized_idx, zeitfenster_field.value)
runtime_ms = int((time.time() - start)*1000)
print(f"Berechnet! Gesamtminuten: {total_time:.1f}, Berechnungszeit: {runtime_ms} ms")
route_df

In [ ]:
# Abschnitt 5: QAOA für Route optimieren und Ergebnis von IBM holen

## Für diesen Abschnitt ist Qiskit Optimization erforderlich


## (Hinweis: Für "echte" große Routingprobleme ist Mapping komplex – hier Mini-TSP für max 20 Punkte!)

def create_tsp_problem(points, depot_idx, prioritized_idx, zeitfenster_minutes):
## Hier: Mini-TSP-Erstellung mit Zeit-/Prioritäts-Constraint
problem = QuadraticProgram()
## [Hier kommt je nach Ansatz das Qiskit Optimization Routing-Problem rein…]
## Siehe Qiskit-Dokumentation für Details!
return problem
API_TOKEN = "DEIN_API_TOKEN"
INSTANCE = "DEIN_INSTANCE"
CHANNEL = "ibm_quantum_platform"
BACKEND = "ibm_oslo" # Beispiel
service = QiskitRuntimeService(
channel=CHANNEL, token=API_TOKEN, instance=INSTANCE
)
tsp_problem = create_tsp_problem(points, depot_idx, prioritized_idx, zeitfenster_field.value)
optimizer = MinimumEigenOptimizer(Sampler(service=service, backend=BACKEND))


## Ausführen und messen 

start = time.time()
result = optimizer.solve(tsp_problem)
runtime_qaoa_ms = int((time.time()-start)*1000)
route_qaoa = result.x # Muss ggf. in die Indexreihenfolge/Route umgewandelt werden
total_qaoa_time = result.fval # oder berechne echte Minuten wie oben

## Speichern
route_qaoa_df = pd.DataFrame({'Index': route_qaoa}) # oder als echte Reihenfolge
route_qaoa_df.to_csv('qaoa_route.csv', index=False)
print(f"QAOA-Job abgeschlossen! Berechnungszeit: {runtime_qaoa_ms} ms")


##💡 Das Mapping vom Problem auf QAOA ist der schwierigste Teil, kann für Mini-TSP/Kundensequenz sprichwörtlich übernommen werden! 💡 

In [ ]:
# Abschnitt 6: Ergebnisse laden und Routen plotten


route_df = pd.read_csv('classical_route.csv')
route_qaoa_df = pd.read_csv('qaoa_route.csv')
def plot_routes(points, route_indices, route_qaoa_indices, depot_idx, prioritized_idx):
plt.figure(figsize=(9,9))
plt.scatter(points[1:,0], points[1:,1], label='Kunden', color='blue')
plt.scatter(points[depot_idx,0], points[depot_idx,1], s=120, marker='s', color='red', label='Depot')
plt.scatter(points[prioritized_idx,0], points[prioritized_idx,1], s=120, marker='*', color='orange', label='Priorisierter Kunde')
for idx in range(len(points)):
plt.text(points[idx,0], points[idx,1], str(idx), fontsize=9, ha='right', va='bottom')
## Klassische Route zeichnen (dünne Pfeile)
for i in range(len(route_indices)-1):
    plt.arrow(points[route_indices[i],0], points[route_indices[i],1],
              points[route_indices[i+1],0]-points[route_indices[i],0],
              points[route_indices[i+1],1]-points[route_indices[i],1],
              color='green', width=0.2, length_includes_head=True, head_width=2, alpha=0.5)
## QAOA-Route zeichnen (andersfarbig)
for i in range(len(route_qaoa_indices)-1):
    plt.arrow(points[route_qaoa_indices[i],0], points[route_qaoa_indices[i],1],
              points[route_qaoa_indices[i+1],0]-points[route_qaoa_indices[i],0],
              points[route_qaoa_indices[i+1],1]-points[route_qaoa_indices[i],1],
              color='purple', width=0.2, length_includes_head=True, head_width=2, alpha=0.5)
plt.title('Routenvergleich: Klassisch vs. QAOA (Quanten)')
plt.legend()
plt.grid(True)
plt.show()

plot_routes(points, route_df['Index'].tolist(), route_qaoa_df['Index'].tolist(), depot_idx, prioritized_idx)


In [ ]:
# Abschnitt 7: Vergleich und Performance-Analyse

print(f"Klassischer Algorithmus: Fahrtzeit genutzt = {total_time:.1f} Min, Berechnungszeit = {runtime_ms} ms")
print(f"Quanten-QAOA-Algorithmus: Fahrtzeit genutzt = {total_qaoa_time:.1f} Min, Berechnungszeit = {runtime_qaoa_ms} ms")
if abs(total_qaoa_time-MAX_MINUTES) < abs(total_time-MAX_MINUTES):
print("🔮 QAOA (Quanten) nutzt die verfügbare Zeit besser aus!")
else:
print("🤖 Klassischer Algorithmus nutzt die Zeit besser – aber je nach Instanz kann sich das ändern!")

#💡💡   Hinweise für dich  💡💡
#💡Nutze jeden Block einzeln in deinem Jupyter Notebook. Passe `API_TOKEN`, `INSTANCE`, und `BACKEND` für den QAOA-Teil an!
#💡Das QAOA-Mapping (QuadraticProgram) ist das mathematisch aufwendigste Segment. Dafür gibt es Qiskit-TSP-Beispiele als Vorlage.
#💡Die Routen werden als CSV gespeichert und dann direkt wieder visualisiert.
#💡Die Visualisierung zeigt Pfeile, Depot, Kunde und Prioritätskunde und ist farbkodiert.
